In [29]:
!pip install --upgrade google-generativeai
!pip install SpeechRecognition pydub google-generativeai

import io
import base64
import speech_recognition as sr
from google.colab import output
from IPython.display import Javascript
import google.generativeai as genai
from google.colab import userdata
from pydub import AudioSegment # Import AudioSegment from pydub

# Configure Gemini API
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)


# ✅ Function to record audio from microphone
def record_audio(sec=5):
    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const recorder = new MediaRecorder(stream);
        let chunks = [];

        recorder.ondataavailable = e => chunks.push(e.data);
        recorder.start();

        await new Promise(resolve => setTimeout(resolve, {sec*1000}));
        recorder.stop();

        await new Promise(resolve => recorder.onstop = resolve);

        let blob = new Blob(chunks);
        let arrayBuffer = await blob.arrayBuffer();
        let base64String = btoa(String.fromCharCode(...new Uint8Array(arrayBuffer)));
        return base64String;
    }}
    recordAudio()
    """
    audio_bytes = output.eval_js(js_code)   # ✅ pass raw JS string, not Javascript()
    audio = base64.b64decode(audio_bytes)
    return io.BytesIO(audio)


# ✅ Convert voice to text
def voice_to_text(sec=5):
    audio_file = record_audio(sec=sec)
    recognizer = sr.Recognizer()

    # Convert the recorded audio to WAV format using pydub
    try:
        audio_segment = AudioSegment.from_file(audio_file, format="webm") # Assuming webm format from browser recording
        wav_file = io.BytesIO()
        audio_segment.export(wav_file, format="wav")
        wav_file.seek(0) # Rewind the BytesIO object
    except Exception as e:
        return f"Error converting audio: {e}"


    with sr.AudioFile(wav_file) as source:
        audio_data = recognizer.record(source)

    try:
        text = recognizer.recognize_google(audio_data)  # free Google Web Speech API
        return text
    except sr.UnknownValueError:
        return "Could not understand audio"
    except sr.RequestError:
        return "Speech Recognition API unavailable"

In [30]:

# ✅ Translate + Summarize using Gemini
def translate_and_summarize(text, target_language="English"):
    model = genai.GenerativeModel('gemini-2.5-flash-preview-05-20')

    # Step 1: Translation
    translation_prompt = f"""
    You are a professional translator. Translate the following text into {target_language}.
    Make sure the output is fully in {target_language}, even if some parts are already in English.

    Text:
    {text}
    """
    translation_response = model.generate_content(translation_prompt)
    translated_text = translation_response.text.strip()

    # Step 2: Summarization
    summarization_prompt = f"""
    You are a summarizer. Summarize the following text in {target_language}.
    Give the summary in short bullet points.

    Text:
    {translated_text}
    """
    summary_response = model.generate_content(summarization_prompt)
    summary_text = summary_response.text.strip()

    return translated_text, summary_text




In [32]:
# ✅ Run the pipeline
print("🎤 Speak now (5 seconds)...")
spoken_text = voice_to_text(sec=5)
print("\nRecognized Speech:\n", spoken_text)

translation, summary = translate_and_summarize(spoken_text, target_language="English")

print("\nTranslated Text:\n", translation)
print("\n" + "="*60 + "\n")
print("Summarization:\n", summary)

🎤 Speak now (5 seconds)...

Recognized Speech:
 Could not understand audio

Translated Text:
 Could not understand audio


Summarization:
 Here's the summary of the text:

*   The audio was not understood.
*   Inability to comprehend the audio.
